Было решено отсечь все классы, у которых нет аннотации. Тк даже если нам не нужна аннотация для обучения детекции, мы используем ее для очистки датасета. Также как правило у классов без аннотации было мало данных.

Остается 
Всего классов: 18
Всего изображений: 18992
Среднее фото на класс: 1055.1

но аннотации есть только у 6752 фото. придется смиритьяс с потерей половины датасета

Метрика: macro F1 <br>
Функция потерь: CrossEntropyLoss<br>
WeightedRandomSampler 

In [20]:
import os
import numpy as np
import torch
import csv
import cv2
import shutil
import random
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import gc

from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report 
from sklearn.model_selection import train_test_split
from torchvision import transforms, datasets, models
from pathlib import Path
from imagededup.methods import PHash
from collections import defaultdict

In [21]:
!rm -r "/mnt/data/archive/simpsons_cleaned/"
!rm -r "/mnt/data/archive/simpsons_split/"

In [ ]:
ANNOTATION_FILE = "/mnt/data/archive/annotation.txt"
ORIGINAL_ROOT = "/mnt/data/archive/simpsons_dataset/"
CLEANED_ROOT = "/mnt/data/archive/simpsons_cleaned/"

THRESHOLD = 10  # Порог pHash
CROP_SIZE = 224


def parse_annotation(path):
    """
    Читает аннотации и группирует по классам.
    Возвращает: {class_name: [{'path': ..., 'bbox': (x1,y1,x2,y2)}, ...]}
    """
    class_groups = defaultdict(list)

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 6:
                continue

            # Исправляем путь
            img_path = row[0].replace("./characters/", "")
            full_path = os.path.join(ORIGINAL_ROOT, img_path)

            try:
                x1 = int(float(row[1]))
                y1 = int(float(row[2]))
                x2 = int(float(row[3]))
                y2 = int(float(row[4]))
            except:
                continue

            class_name = row[5].strip()

            if os.path.exists(full_path):
                #группируем по персонажам 
                class_groups[class_name].append(
                    {"path": full_path, "bbox": (x1, y1, x2, y2)}
                )

    return class_groups


def crop_character(img_path, bbox, crop_size=224):
    """
    Вырезает персонажа по боксу и ресайзит для сравнения pHash. Оригинальное изображение не меняется
    """
    img = cv2.imread(img_path)
    if img is None:
        return None

    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox

    # Защита от выхода за границы
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    if x2 <= x1 or y2 <= y1:
        return None

    # Кроп + ресайз для сравнения
    crop = img[y1:y2, x1:x2]
    crop = cv2.resize(crop, (crop_size, crop_size))

    return crop


def find_unique_images(items, class_name, crop_size=224):
    """
    Находит уникальные изображ0ения внутри класса.Возвращает список путей к оригиналам
    """
    # Временная папка для кропов
    temp_dir = f"./temp_crops_{class_name.replace('/', '_')}"
    os.makedirs(temp_dir, exist_ok=True)

    valid_items = []

    # Создаём кропы для сравнения
    for i, item in enumerate(items):
        crop = crop_character(item["path"], item["bbox"], crop_size)
        if crop is not None:
            save_name = f"crop_{i}.jpg"
            save_path = os.path.join(temp_dir, save_name)
            cv2.imwrite(save_path, crop)
            valid_items.append(
                {"original_path": item["path"], "crop_path": save_path, "index": i}
            )

    # пароверка на минимальное количество изображений
    if len(valid_items) < 2:
        shutil.rmtree(temp_dir, ignore_errors=True)  # удаляет временную папку с кропами
        return [item["original_path"] for item in valid_items]

    # запускаем pHash на кропах
    phasher = PHash(verbose=False)
    encodings = phasher.encode_images(image_dir=temp_dir)  #возвращает словарь {имя файла: хеш}
    duplicates = phasher.find_duplicates(encoding_map=encodings, max_distance_threshold=THRESHOLD, scores=False)
    # возвращает оригнал : [{дубли}]

    # Собираем индексы дублей которые будем удалять
    duplicate_indices = set()
    processed = set()

    for img_name, dup_list in duplicates.items():
        if img_name in processed:
            continue

        processed.add(img_name)

        # Оставляем первый  остальные помечаем как дубли
        for dup_name in dup_list:
            if dup_name == "" or dup_name in processed:
                continue
            processed.add(dup_name)

            # Извлекаем индекс из имени файла crop_X.jpg
            try:
                idx = int(dup_name.split("_")[1].split(".")[0])
                duplicate_indices.add(idx)
            except:
                continue

    # Чистим временную папку
    shutil.rmtree(temp_dir, ignore_errors=True)

    # Возвращаем пути к уникальным оригиналам
    unique_paths = [
        item["original_path"]
        for item in valid_items
        if item["index"] not in duplicate_indices
    ]

    return unique_paths


def copy_to_cleaned_dataset(unique_paths, class_name):
    """
    Копирует оригинальные изображения в новую папку.
    Сохраняет структуру папок.
    """
    # Создаём папку класса в очищенном датасете
    dest_class_folder = os.path.join(CLEANED_ROOT, class_name)
    os.makedirs(dest_class_folder, exist_ok=True)

    copied_count = 0
    for orig_path in unique_paths:
        # Имя файла остаётся тем же
        filename = os.path.basename(orig_path)
        dest_path = os.path.join(dest_class_folder, filename)

        # Копируем оригинал
        shutil.copy2(orig_path, dest_path)
        copied_count += 1

    return copied_count


if __name__ == "__main__":

    # создаём корневую папку очищенного датасета
    os.makedirs(CLEANED_ROOT, exist_ok=True)

    groups = parse_annotation(ANNOTATION_FILE)

    total_original = sum(len(items) for items in groups.values())

    # Статистика
    stats = []
    total_unique = 0

    # Обработка каждого класса
    for class_name, items in groups.items():
        # Находим уникальные изображения
        unique_paths = find_unique_images(items, class_name, CROP_SIZE)

        # Копируем в новый датасет
        copied = copy_to_cleaned_dataset(unique_paths, class_name)
        total_unique += copied

        stats.append(
            {
                "class": class_name,
                "original": len(items),
                "unique": copied,
                "removed": len(items) - copied,
            }
        )

In [23]:
# статистикка

MAIN_DATASET_PATH = "/mnt/data/archive/simpsons_cleaned/"

def count_images_in_folders(base_path):
    stats = {}
    path = Path(base_path)

    if not path.exists():
        print(f"Папка {base_path} не найдена!")
        return stats

    for folder in path.iterdir():
        if folder.is_dir():
            images = [
                f
                for f in folder.iterdir()
                if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
            ]
            stats[folder.name] = len(images)

    return stats


def print_statistics(stats, dataset_name):
    if not stats:
        print("Нет данных для анализа")
        return

    values = list(stats.values())
    total_images = sum(values)
    total_classes = len(stats)
    min_count = min(values)
    max_count = max(values)
    mean_count = sum(values) / len(values)

    min_class = [k for k, v in stats.items() if v == min_count]
    max_class = [k for k, v in stats.items() if v == max_count]

    print(f"Всего классов: {total_classes}")
    print(f"Всего изображений: {total_images}")
    print(f"Среднее фото на класс: {mean_count:.1f}")
    print(f"Минимум фото в классе: {min_count} ({', '.join(min_class)})")
    print(f"Максимум фото в классе: {max_count} ({', '.join(max_class)})")

    for cls, count in sorted(stats.items(), key=lambda x: x[1], reverse=True)[::]:
        print(f"   -{cls}: {count} фото")


if __name__ == "__main__":
    train_stats = count_images_in_folders(MAIN_DATASET_PATH)
    print_statistics(train_stats, "dataset")

Всего классов: 18
Всего изображений: 6489
Среднее фото на класс: 360.5
Минимум фото в классе: 163 (sideshow_bob)
Максимум фото в классе: 624 (charles_montgomery_burns)
   -charles_montgomery_burns: 624 фото
   -homer_simpson: 604 фото
   -abraham_grampa_simpson: 564 фото
   -ned_flanders: 562 фото
   -lisa_simpson: 543 фото
   -marge_simpson: 540 фото
   -bart_simpson: 536 фото
   -principal_skinner: 487 фото
   -krusty_the_clown: 222 фото
   -nelson_muntz: 216 фото
   -kent_brockman: 209 фото
   -moe_szyslak: 208 фото
   -milhouse_van_houten: 204 фото
   -chief_wiggum: 204 фото
   -edna_krabappel: 204 фото
   -apu_nahasapeemapetilon: 202 фото
   -comic_book_guy: 197 фото
   -sideshow_bob: 163 фото


In [24]:
CLEANED_ROOT = "/mnt/data/archive/simpsons_cleaned/"
SPLIT_ROOT = "/mnt/data/archive/simpsons_split/"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42  

def get_class_images(root_dir):
    """
    Собирает пути к изображениям, сгруппированные по классам {class_name: [path1, path2, ...]}
    """
    class_images = defaultdict(list)

    for class_name in os.listdir(root_dir):
        class_path = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        for filename in os.listdir(class_path):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                img_path = os.path.join(class_path, filename)
                class_images[class_name].append(img_path)

    return class_images


def stratified_split(class_images, train_ratio, val_ratio, test_ratio, seed=42):
    """
    Делает стратифицированное разбиение для каждого класса отдельно. Возвращает: {split_name: {class_name: [paths]}}
    """
    splits = {"train": {}, "val": {}, "test": {}}

    random.seed(seed)

    for class_name, images in class_images.items():
        # перемешиваем внутри класса
        shuffled = images.copy()
        random.shuffle(shuffled)

        train_imgs, temp_imgs = train_test_split(
            shuffled,
            train_size=train_ratio,
            random_state=seed,
            shuffle=True,  # на всякий случай
        )

        #  val vs test из оставшихся
        # нормируем пропорции относительно оставшейся части
        val_size = val_ratio / (val_ratio + test_ratio)
        val_imgs, test_imgs = train_test_split(temp_imgs, train_size=val_size, random_state=seed, shuffle=True)

        splits["train"][class_name] = train_imgs
        splits["val"][class_name] = val_imgs
        splits["test"][class_name] = test_imgs

    return splits


def copy_split_images(splits, output_root):
    """
    Копирует изображения в новую структуру папок output_root/{train,val,test}/{class_name}/{filename}
    """
    stats = {}

    for split_name, class_dict in splits.items():
        split_path = os.path.join(output_root, split_name)
        os.makedirs(split_path, exist_ok=True)

        split_stats = {}

        for class_name, img_paths in class_dict.items():
            class_folder = os.path.join(split_path, class_name)
            os.makedirs(class_folder, exist_ok=True)

            for img_path in img_paths:
                filename = os.path.basename(img_path)
                dest_path = os.path.join(class_folder, filename)
                shutil.copy2(img_path, dest_path)

            split_stats[class_name] = len(img_paths)

        stats[split_name] = split_stats
        print(f"{split_name.upper()}: {sum(split_stats.values())} изображений")

    return stats

if __name__ == "__main__":
    class_images = get_class_images(CLEANED_ROOT)
    total_images = sum(len(imgs) for imgs in class_images.values())

    splits = stratified_split(
        class_images, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=RANDOM_SEED
    )

    stats = copy_split_images(splits, SPLIT_ROOT)

TRAIN: 4533 изображений
VAL: 973 изображений
TEST: 983 изображений


In [25]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    DEVICE = torch.device("cuda:0")
    torch.backends.cudnn.benchmark = (True)

In [26]:
DATA_ROOT = "/mnt/data/archive/simpsons_split/"
MODEL_PATH = "/mnt/data/archive/best_resnet34_simpsons.pth"

NUM_CLASSES = 18
BATCH_SIZE = 64  
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 10

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # среднее и стандартное отклонение для изображений в imagenet
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=val_transform)

# подсчет изображений по классам
class_counts = np.zeros(NUM_CLASSES)
for _, label in train_dataset:
    class_counts[label] += 1

class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for _, label in train_dataset]
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

class_names = train_dataset.classes

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Classes: {train_dataset.classes}", {len(train_dataset.classes)})

Train batches: 71, Val batches: 16
Classes: ['abraham_grampa_simpson', 'apu_nahasapeemapetilon', 'bart_simpson', 'charles_montgomery_burns', 'chief_wiggum', 'comic_book_guy', 'edna_krabappel', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lisa_simpson', 'marge_simpson', 'milhouse_van_houten', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'principal_skinner', 'sideshow_bob'] {18}


ResNet34

In [28]:
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

In [ ]:
# заморозить все слои чтобы предотвратить их обновление
for param in model.parameters():
    param.requires_grad = False

# разморозить layer4 + FC чтобы дообучить их на симпсонах
# layer4 потому что симпсоны не очень похожи на ImageNet
for param in model.layer4.parameters():
    param.requires_grad = True

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, NUM_CLASSES)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss().to(DEVICE)

optimizer = optim.Adam(
    [
        {
            "params": model.layer4.parameters(),
            "lr": LEARNING_RATE * 0.1,  
        },
        {
            "params": model.fc.parameters(), 
            "lr": LEARNING_RATE
        },
    ],
    weight_decay=1e-4,
)

#todo
#добавить возможность классифицировать другие классы не симпсонов и других симпсонов
#out of domain и добавить нового симпсона

def calculate_macro_f1(predictions, labels):
    """Считает Macro F1-score"""
    return f1_score(labels, predictions, average="macro")


def evaluate(model, loader, device):
    """Оценка модели"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating", leave=False):
            # Перемещаем данные на GPU
            inputs = inputs.to(device, non_blocking=True) 
            labels = labels.to(device, non_blocking=True)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return calculate_macro_f1(all_preds, all_labels)

best_val_f1 = 0.0
patience_counter = 0
history = {"train_loss": [], "val_f1": []}


for epoch in range(NUM_EPOCHS):
    print(f"Эпоха {epoch+1}/{NUM_EPOCHS}")
    # train
    model.train()
    running_loss = 0.0 #накопленная сумма потерь за всю эпоху обучения

    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        # Перемещаем данные на GPU
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.set_grad_enabled(True):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    # вот из-за двух предыдущих строк получается взвешенное среднее по изобраэениям
    # чтобы компенсировать возможность что последний батч неполный
    history["train_loss"].append(epoch_train_loss)

    # val
    val_f1 = evaluate(model, val_loader, DEVICE)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {epoch_train_loss:.4f} | Val Macro F1: {val_f1:.4f}")

    # early stop and checkoint

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": val_f1,
                "class_names": class_names,
            },
            MODEL_PATH,
        )
        print(f"Новая лучшая модель сохранена (F1: {val_f1:.4f})")
    else:
        patience_counter += 1
        print(f" Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping на эпохе {epoch+1}")
            break

print(f" Лучший Val Macro F1: {best_val_f1:.4f}")

# test

checkpoint = torch.load(MODEL_PATH)
model.load_state_dict(checkpoint["model_state_dict"])

test_f1 = evaluate(model, test_loader, DEVICE)
print(f"Test Macro F1: {test_f1:.4f}")

print("\nClassification Report (Test):")
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

Эпоха 1/50


Train Loss: 1.3395 | Val Macro F1: 0.8588
Новая лучшая модель сохранена (F1: 0.8588)
Эпоха 2/50


Train Loss: 0.4106 | Val Macro F1: 0.8916
Новая лучшая модель сохранена (F1: 0.8916)
Эпоха 3/50


Train Loss: 0.2406 | Val Macro F1: 0.9060
Новая лучшая модель сохранена (F1: 0.9060)
Эпоха 4/50


Train Loss: 0.1811 | Val Macro F1: 0.9008
 Patience: 1/10
Эпоха 5/50


...

Train Loss: 0.0379 | Val Macro F1: 0.9105
 Patience: 8/10
Эпоха 21/50


Train Loss: 0.0283 | Val Macro F1: 0.9256
 Patience: 9/10
Эпоха 22/50


Train Loss: 0.0251 | Val Macro F1: 0.9309
 Patience: 10/10

Early stopping на эпохе 22
 Лучший Val Macro F1: 0.9332


Test Macro F1: 0.9322

Classification Report (Test):


                          precision    recall  f1-score   support

  abraham_grampa_simpson     0.9651    0.9765    0.9708        85
  apu_nahasapeemapetilon     1.0000    0.9677    0.9836        31
            bart_simpson     0.9398    0.9630    0.9512        81
charles_montgomery_burns     0.8911    0.9574    0.9231        94
            chief_wiggum     0.9032    0.9032    0.9032        31
          comic_book_guy     0.9032    0.9333    0.9180        30
          edna_krabappel     0.8485    0.9032    0.8750        31
           homer_simpson     0.9205    0.8901    0.9050        91
           kent_brockman     0.9143    1.0000    0.9552        32
        krusty_the_clown     0.9118    0.9118    0.9118        34
            lisa_simpson     0.9630    0.9512    0.9571        82
           marge_simpson     0.9747    0.9506    0.9625        81
     milhouse_van_houten     0.8571    0.9677    0.9091        31
             moe_szyslak     0.9655    0.8750    0.9180        32
         